In [376]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import pandas as pd
import numpy as  np
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedShuffleSplit
MODEL_FILE='model.pkl'
PIPELINE_FILE='pipeline.pkl'

In [377]:
def bp(num,cat):
    numpipeline=Pipeline([("imputer",SimpleImputer(strategy='median')),("scaler",StandardScaler())])
    catpipeline=Pipeline([("onehot",OneHotEncoder(handle_unknown='ignore'))])
    pipeline=ColumnTransformer([('num',numpipeline,num),('cat',catpipeline,cat)])
    return pipeline

In [378]:
if not os.path.exists(MODEL_FILE):
    df= pd.read_csv("/Users/mohammedzaheeruddin/Downloads/housing.csv")
    df['icat']=pd.cut(df['median_income'],bins=[0,1.5,3.0,4.5,6.0,np.inf],labels=[1,2,3,4,5])
    shuff=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=72)
    for tr_,_ in shuff.split(df,df['icat']):
        tr=df.loc[tr_].drop('icat',axis=1)
    housing=tr.copy()
    hlabels=housing['median_house_value'].copy()
    housing=housing.drop('median_house_value',axis=1)
    num=housing.drop('ocean_proximity',axis=1).columns.tolist()
    cat=['ocean_proximity']
    pipeline=bp(num,cat)
    Hprep=pipeline.fit_transform(housing)
    model=RandomForestRegressor(random_state=72)
    model.fit(Hprep,hlabels)
    joblib.dump(model, MODEL_FILE)
    joblib.dump(pipeline, PIPELINE_FILE)
else:
    model = joblib.load(MODEL_FILE)
    pipeline = joblib.load(PIPELINE_FILE)
    input_data =pd.read_csv("/Users/mohammedzaheeruddin/Downloads/housing.csv")
    transformed_input = pipeline.transform(input_data)
    predictions = model.predict(transformed_input)
    input_data["median_house_value"] = predictions
    input_data.to_csv("output.csv", index=False)
    print("Inference complete. Results saved to output.csv")
        

Inference complete. Results saved to output.csv
